# **Summarize Video with Gemini (CrewAI Tool)**

This notebook shows how to create a **CrewAI Tool** that uses the official
[Google GenAI SDK](https://pypi.org/project/google-genai/) to summarize videos
(e.g. YouTube URLs).

The tool:
- Accepts a video URL + prompt
- Calls Gemini (AI Studio) with `video/*` file data
- Returns a Markdown summary
- Supports both streaming and non-streaming modes
- Can be attached to a CrewAI Agent

## 1. Install & Setup

In [1]:
%pip install -q -U google-genai crewai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 13.4 MB/s eta 0:00:00


In [ ]:
import os

# set your Gemini API key (AI Studio, not Vertex)
# get one here: https://aistudio.google.com/app/apikey
os.environ["GEMINI_API_KEY"] = "AIza..."  # replace with your key

## 2. Define the Tool

In [7]:
import os
from typing import Optional
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from crewai.tools import BaseTool

def _summarize_video_with_genai(
    video_url: str,
    prompt: str,
    model: str,
    stream: bool,
    mime_type: str,
    response_mime_type: str,
) -> str:
    """Internal helper that calls the Google AI Studio SDK."""
    api_key = GEMINI_API_KEY
    if not api_key:
        return "Missing GEMINI_API_KEY. Please set it first."

    client = genai.Client(api_key=api_key)

    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part(
                    file_data=types.FileData(
                        file_uri=video_url,
                        mime_type=mime_type,
                    )
                ),
                types.Part.from_text(text=prompt),
            ],
        )
    ]
    cfg = types.GenerateContentConfig(response_mime_type=response_mime_type)

    try:
        if stream:
            chunks = []
            for chunk in client.models.generate_content_stream(
                model=model, contents=contents, config=cfg
            ):
                if getattr(chunk, "text", None):
                    chunks.append(chunk.text)
            return ("".join(chunks)).strip() or "No text returned."
        else:
            resp = client.models.generate_content(
                model=model, contents=contents, config=cfg
            )
            return (getattr(resp, "text", "") or "").strip() or "No text returned."
    except Exception as e:
        return f"Gemini error: {e}"


class VideoSummaryArgs(BaseModel):
    video_url: str = Field(description="YouTube or direct video URL")
    prompt: Optional[str] = Field(
        default="Generate a detailed summary with key points and timestamps (if available).",
        description="Instruction for how to summarize"
    )
    stream: Optional[bool] = Field(default=False, description="If True, uses server-side streaming")
    model: Optional[str] = Field(default="gemini-2.5-pro", description="Gemini model name")
    mime_type: Optional[str] = Field(default="video/*", description="Video MIME type")
    response_mime_type: Optional[str] = Field(default="text/plain", description="Response format")


class SummarizeVideoGeminiTool(BaseTool):
    name: str = "summarize_video_gemini"
    description: str = """
    Summarize a video using Gemini (Google AI Studio).

    Args:
        video_url: A YouTube or direct video URL.
        prompt:    Instruction for how to summarize.
        stream:    If True, uses server-side streaming.
        model:     Gemini model name, e.g. 'gemini-2.5-pro'.
        mime_type: Usually 'video/*'.
        response_mime_type: 'text/plain' only (not 'text/markdown').

    Returns:
        Plain text summary.
    """
    args_schema: type[BaseModel] = VideoSummaryArgs

    def _run(
        self,
        video_url: str,
        prompt: str = "Generate a detailed summary with key points and timestamps (if available).",
        stream: bool = False,
        model: str = "gemini-2.5-pro",
        mime_type: str = "video/*",
        response_mime_type: str = "text/plain",
    ) -> str:
        return _summarize_video_with_genai(
            video_url=video_url,
            prompt=prompt,
            model=model,
            stream=stream,
            mime_type=mime_type,
            response_mime_type=response_mime_type,
        )

## 3. Direct Test

In [8]:
# quick sanity check (direct call)
summary = SummarizeVideoGeminiTool().run(
    video_url="https://www.youtube.com/watch?v=e-LNrKZMxOk",
    stream=True
)
display(summary)

Using Tool: summarize_video_gemini


'Here is a detailed summary of the key topics discussed in the video clip.\n\n### Topic 1: Meta\'s "Boom-Bust" AI Strategy\n**Timestamp: 00:00 - 00:52**\nJason Calacanis introduces the topic by highlighting a rapid and contradictory shift in Meta\'s AI strategy. Just weeks after a "blockbuster spending spree"—which included trying to acquire Ilya Sutskever\'s $32 billion startup, poaching top talent like Daniel Gross, investing $14.3 billion in Scale AI, and reportedly offering $100 million bonuses to OpenAI staff—Meta is now reportedly freezing AI hiring and considering downsizing its AI division. Calacanis questions this intense "boom-bust cycle" happening in such a compressed timeframe.\n\n### Topic 2: The AI "Boom" and Correction\n**Timestamp: 00:52 - 02:07**\nDavid Sacks analyzes the situation, explaining that the recent frenzy was a classic "boom cycle" phenomenon.\n*   **Insane Offers:** He notes that multi-billion dollar acquisition offers for pre-product startups and nine-figu

## 4. Use Inside a CrewAI Agent

In [9]:
from crewai import Agent, LLM

llm = LLM(model="gemini/gemini-2.5-pro", api_key=GEMINI_API_KEY)

video_analyst = Agent(
    role="Video Analyst",
    goal="Extract accurate, detailed summaries and insights from videos.",
    backstory="Understands longform content and produces structured briefings.",
    tools=[SummarizeVideoGeminiTool()],
    llm=llm,
    verbose=True,
)

try:
    result = video_analyst.kickoff(messages="Summarize this video with action items: https://www.youtube.com/watch?v=hBEAF-pCkz4")
except Exception as e:
    print(f"Error: {e}")

╭─────────────────────────────────────────────── LiteAgent Started ───────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Session Started                                                                                      │
│  Name: Video Analyst                                                                                            │
│  id: 3ca70b7f-73f8-40ce-b71d-6c1c0bfb27ff                                                                       │
│  role: Video Analyst                                                                                            │
│  goal: Extract accurate, detailed summaries and insights from videos.                                           │
│  backstory: Understands longform content and produces structured briefings.                                     │
│  tools: [CrewStructuredTool(name='summarize_video_gemini', description='Tool Name: summarize_video_gemini       │
│  Tool Arguments: {'video_url': {'description': 'YouTube or direct video URL', 'type': 'str'}, 'prompt':         │
│  {'description': 'Instruction for how to summarize', 'type': 'Union[str, NoneType]'}, 'stream':                 │
│  {'description': 'If True, uses server-side streaming', 'type': 'Union[bool, NoneType]'}, 'model':              │
│  {'description': 'Gemini model name', 'type': 'Union[str, NoneType]'}, 'mime_type': {'description': 'Video      │
│  MIME type', 'type': 'Union[str, NoneType]'}, 'response_mime_type': {'description': 'Response format', 'type':  │
│  'Union[str, NoneType]'}}                                                                                       │
│  Tool Description:                                                                                              │
│      Summarize a video using Gemini (Google AI Studio).                                                         │
│                                                                                                                 │
│      Args:                                                                                                      │
│          video_url: A YouTube or direct video URL.                                                              │
│          prompt:    Instruction for how to summarize.                                                           │
│          stream:    If True, uses server-side streaming.                                                        │
│          model:     Gemini model name, e.g. 'gemini-2.5-pro'.                                                   │
│          mime_type: Usually 'video/*'.                                                                          │
│          response_mime_type: 'text/plain' only (not 'text/markdown').                                           │
│                                                                                                                 │
│      Returns:                                                                                                   │
│          Plain text summary.                                                                                    │
│      ')]                                                                                                        │
│  verbose: True                                                                                                  │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Video Analyst                                                                                           │
│                                                                                                                 │
│  Thought: Thought: The user wants me to summarize a YouTube video and provide a list of action items. The       │
│  `summarize_video_gemini` tool is the only available tool and it is perfectly suited for this task. I need to   │
│  call this tool with the `video_url` provided by the user and a `prompt` that instructs the model to generate   │
│  a summary and a list of action items.The user wants a summary and action items from a YouTube video. I have    │
│  the `summarize_video_gemini` tool which can take a video URL and a prompt to produce a summary. I will use     │
│  this tool to fulfill the user's request.                                                                       │
│                                                                                                                 │
│  Using Tool: summarize_video_gemini                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"video_url\": \"https://www.youtube.com/watch?v=hBEAF-pCkz4\", \"prompt\": \"Summarize the video and        │
│  provide a list of action items.\"}"                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Of course. Here is a summary of the video's key topics and a list of action items mentioned.                   │
│                                                                                                                 │
│  ### Video Summary                                                                                              │
│                                                                                                                 │
│  This video provides an update on several major developments in the world of artificial intelligence. Key       │
│  highlights include:                                                                                            │
│                                                                                                                 │
│  *   **Elon Musk's "Macrohard":** Elon Musk and his company, xAI, have announced a new project called           │
│  "Macrohard," a tongue-in-cheek name for a serious endeavor to build a purely AI-based software company,        │
│  conceptualized as an end-to-end neural network operating system that could simulate companies like Microsoft.  │
│  *   **AI-Generated Worlds & Games:** The concept of AI-generated content is rapidly becoming a reality. The    │
│  video showcases "Mirage 2" from Dynamics Lab, a new, playable online engine that can turn any image into a     │
│  live, interactive 3D world. This follows similar developments like Google's Genie 3.                           │
│  *   **Major Partnerships:**                                                                                    │
│      *   **Meta & Midjourney:** Meta is partnering with Midjourney to license its advanced "aesthetic           │
│  technology," likely to improve the image generation capabilities of Meta's future products.                    │
│      *   **Meta & Google:** Meta has struck a $10 billion, six-year deal to use Google Cloud's infrastructure   │
│  to support its massive AI expansion efforts.                                                                   │
│  *   **Robotics Advancements:**                                                                                 │
│      *   **Figure Robot:** A new demo shows the Figure AI robot successfully folding towels, a complex task     │
│  that demonstrates its advanced dexterity. This is achieved through an end-to-end neural network, not           │
│  pre-programming, showing significant progress toward capable personal chore robots.                            │
│      *   **Nvidia Jetson Thor:** Nvidia has unveiled Jetson Thor, a new, powerful computer chipset              │
│  specifically designed for the next generation of general and humanoid robots.                                  │
│  *   **Industry & Politics:**                                                                                   │
│      *   **AI Super PAC:** Major players in the AI industry, including Andreessen Horowitz and Perplexity,      │
│  have formed a bipartisan super PAC called "Leading the Future" to promote pro-AI innovation policies.          │
│      *   **Uber vs. Tesla:** The ...                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Video Analyst                                                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Summary of the Video                                                                                       │
│                                                                                                                 │
│  The video discusses several significant advancements and news in the artificial intelligence sector:           │
│                                                                                                                 │
│  *   **Elon Musk's "Macrohard":** Elon Musk's company, xAI, is embarking on a new project humorously named      │
│  "Macrohard." The ambitious goal is to create a software company entirely based on AI, envisioned as a neural   │
│  network operating system capable of simulating large tech companies like Microsoft.                            │
│  *   **AI-Generated Interactive Worlds:** The video highlights the emergence of AI that can generate            │
│  interactive 3D worlds from images. It features "Mirage 2" from Dynamics Lab, an online engine that allows      │
│  users to experience this technology firsthand, similar to Google's Genie 3.                                    │
│  *   **Key Industry Partnerships:**                                                                             │
│      *   **Meta and Midjourney:** Meta is set to license Midjourney's "aesthetic technology," likely to         │
│  enhance the image generation capabilities of its products.                                                     │
│      *   **Meta and Google:** A substantial $10 billion, six-year agreement has been signed for Meta to         │
│  utilize Google Cloud's infrastructure to power its AI expansion.                                               │
│  *   **Progress in Robotics:**                                                                                  │
│      *   **Figure AI Robot:** A new demonstration shows the Figure AI robot performing the complex task of      │
│  folding towels using an end-to-end neural network, indicating a significant step towards practical household   │
│  robots.                                                                                                        │
│      *   **Nvidia Jetson Thor:** Nvidia has launched Jetson Thor, a powerful computer chipset designed to be    │
│  the brain for the next generation of humanoid and general-purpose robots.                                      │
│  *   **AI's Influence in Politics and Industry Debates:**                                                       │
│      *   **"Leading the Future" Super PAC:** Key AI industry figures have formed a bipartisan super PAC to      │
│  advocate for policies that promote AI innovation.                                                              │
│      *   **Uber vs. Tesla on Self-Driving:** The video highlights a strategic disagreement between Uber's CEO,  │
│  who believes LiDAR is essential for self-driving cars, and Elon Musk, who argues that a camera-only system is  │
│  safer and that additional sensors can create "sensor contention."                                              │
│  *   **Challenges in AI Adoption:** An MIT report reveals that 95% of generative AI pilot programs in           │
│  businesses are failing, indicating that companies are finding it difficult to identify effective use cases     │
│  and integrate the technology successfully.            

╭───────────────────────────────────────────── LiteAgent Completion ──────────────────────────────────────────────╮
│                                                                                                                 │
│  LiteAgent Completed                                                                                            │
│  Name: Video Analyst                                                                                            │
│  id: 3ca70b7f-73f8-40ce-b71d-6c1c0bfb27ff                                                                       │
│  role: Video Analyst                                                                                            │
│  goal: Extract accurate, detailed summaries and insights from videos.                                           │
│  backstory: Understands longform content and produces structured briefings.                                     │
│  tools: [CrewStructuredTool(name='summarize_video_gemini', description='Tool Name: summarize_video_gemini       │
│  Tool Arguments: {'video_url': {'description': 'YouTube or direct video URL', 'type': 'str'}, 'prompt':         │
│  {'description': 'Instruction for how to summarize', 'type': 'Union[str, NoneType]'}, 'stream':                 │
│  {'description': 'If True, uses server-side streaming', 'type': 'Union[bool, NoneType]'}, 'model':              │
│  {'description': 'Gemini model name', 'type': 'Union[str, NoneType]'}, 'mime_type': {'description': 'Video      │
│  MIME type', 'type': 'Union[str, NoneType]'}, 'response_mime_type': {'description': 'Response format', 'type':  │
│  'Union[str, NoneType]'}}                                                                                       │
│  Tool Description:                                                                                              │
│      Summarize a video using Gemini (Google AI Studio).                                                         │
│                                                                                                                 │
│      Args:                                                                                                      │
│          video_url: A YouTube or direct video URL.                                                              │
│          prompt:    Instruction for how to summarize.                                                           │
│          stream:    If True, uses server-side streaming.                                                        │
│          model:     Gemini model name, e.g. 'gemini-2.5-pro'.                                                   │
│          mime_type: Usually 'video/*'.                                                                          │
│          response_mime_type: 'text/plain' only (not 'text/markdown').                                           │
│                                                                                                                 │
│      Returns:                                                                                                   │
│          Plain text summary.                                                                                    │
│      ')]                                                                                                        │
│  verbose: True                                                                                                  │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Print the video summary

In [12]:
from IPython.display import Markdown
Markdown(result.raw)

### Summary of the Video

The video discusses several significant advancements and news in the artificial intelligence sector:

*   **Elon Musk's "Macrohard":** Elon Musk's company, xAI, is embarking on a new project humorously named "Macrohard." The ambitious goal is to create a software company entirely based on AI, envisioned as a neural network operating system capable of simulating large tech companies like Microsoft.
*   **AI-Generated Interactive Worlds:** The video highlights the emergence of AI that can generate interactive 3D worlds from images. It features "Mirage 2" from Dynamics Lab, an online engine that allows users to experience this technology firsthand, similar to Google's Genie 3.
*   **Key Industry Partnerships:**
    *   **Meta and Midjourney:** Meta is set to license Midjourney's "aesthetic technology," likely to enhance the image generation capabilities of its products.
    *   **Meta and Google:** A substantial $10 billion, six-year agreement has been signed for Meta to utilize Google Cloud's infrastructure to power its AI expansion.
*   **Progress in Robotics:**
    *   **Figure AI Robot:** A new demonstration shows the Figure AI robot performing the complex task of folding towels using an end-to-end neural network, indicating a significant step towards practical household robots.
    *   **Nvidia Jetson Thor:** Nvidia has launched Jetson Thor, a powerful computer chipset designed to be the brain for the next generation of humanoid and general-purpose robots.
*   **AI's Influence in Politics and Industry Debates:**
    *   **"Leading the Future" Super PAC:** Key AI industry figures have formed a bipartisan super PAC to advocate for policies that promote AI innovation.
    *   **Uber vs. Tesla on Self-Driving:** The video highlights a strategic disagreement between Uber's CEO, who believes LiDAR is essential for self-driving cars, and Elon Musk, who argues that a camera-only system is safer and that additional sensors can create "sensor contention."
*   **Challenges in AI Adoption:** An MIT report reveals that 95% of generative AI pilot programs in businesses are failing, indicating that companies are finding it difficult to identify effective use cases and integrate the technology successfully.

### Action Items

Here are some actions you can take based on the information in the video:

1.  **Experience AI-Generated Worlds:** Visit **dynamicslab.ai** to try out the Mirage 2 engine and turn images into playable 3D environments.
2.  **Explore a Career at xAI:** If you're interested in working on the "Macrohard" project, Elon Musk has extended an invitation via his tweet to join xAI.
3.  **Watch an AI Play a Game:** Check the video's description for a link to a live stream of GPT-5 playing and completing the game Pokémon Crystal.
4.  **Enhance Your Marketing with AI:** Download the free "GPT-5 Marketing Stack: 10 Advanced Prompts" guide from HubSpot, available through a link in the video description.
5.  **Learn About AI-Ready Hardware:** For those interested in high-performance computing for AI, the video's description provides a link to learn more about Dell Pro Max workstations equipped with Nvidia GPUs.
6.  **Understand AI Implementation Challenges:** Read the MIT report on the high failure rate of generative AI pilot programs to learn about the common obstacles businesses face.
7.  **Follow AI Policy Developments:** Look into the "Leading the Future" super PAC to stay informed about the political initiatives supporting AI innovation.